# Finance IDC Project — Comprehensive Exploratory Data Analysis (EDA)

**Purpose:** This notebook is a detailed, reproducible investigation of the project's market, news, sentiment, alignment, and modeling data. It is designed to explain not only *what* each chart or statistic says, but also *why* it matters to the research question and what conclusions are or are not justified.

> **Research question:** Do financial-news sentiment and news volume provide useful information for forecasting the next NIFTY 50 session, beyond information already contained in market variables?

This is an exploratory notebook—not a claim that the project has discovered a profitable trading strategy. Descriptive association is not predictive performance, and predictive performance is not evidence of a tradable edge after costs.

## 0. How to use this notebook

1. Place this notebook inside the project repository (recommended: `notebooks/`).
2. Ensure the project has already produced its processed CSV files by running the pipeline.
3. Set `PROJECT_ROOT` below to the repository root if automatic discovery fails.
4. Run cells from top to bottom. Sections are designed to tolerate missing optional files and to print useful diagnostics.
5. Read the interpretation notes; do not treat every correlation or plot as a modeling result.

The notebook does **not** recollect data, retrain the production model, or modify source data. It reads generated files and performs exploratory checks.

## 1. Project logic and analytical boundaries

The project combines two information streams:

- **Market stream:** NIFTY 50 prices/returns and related market variables.
- **News stream:** headline records, article counts, sentiment scores, and daily aggregates.

The streams are useful only if their timestamps and information availability are aligned correctly. A value recorded on calendar date *t* is not automatically available at the NIFTY forecast cutoff on date *t*. In particular, an external market's closing price may occur after the NIFTY session ends. This is a potential look-ahead bias, not a cosmetic timestamp issue.

The notebook therefore separates:
- data quality and descriptive EDA;
- timestamp/alignment audits;
- exploratory relationships;
- model-oriented target and feature checks.

Any final performance claim must come from chronological validation, with preprocessing and feature selection performed inside each training window.

## 2. Environment and imports

In [ ]:
from pathlib import Path
import warnings
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

## 3. Locate the repository and load generated data

In [ ]:
# If automatic discovery chooses the wrong directory, replace this with the path
# to the root folder containing data/ and src/.
CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]
PROJECT_ROOT = next((p for p in CANDIDATES if (p / "data").exists()), Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
print("Project root:", PROJECT_ROOT.resolve())
print("Data directory exists:", DATA_DIR.exists())

# Common project output names. Missing files are reported rather than treated as fatal.
FILES = {
    "model": DATA_DIR / "processed" / "model_dataset.csv",
    "alignment": DATA_DIR / "processed" / "news_market_alignment.csv",
    "daily_news": DATA_DIR / "processed" / "news_market_daily.csv",
    "scored_news": DATA_DIR / "processed" / "news_sentiment_scored.csv",
    "daily_sentiment": DATA_DIR / "processed" / "news_sentiment_daily.csv",
    "sampling_audit": DATA_DIR / "processed" / "sampling_audit.csv",
    "prices": DATA_DIR / "processed" / "nifty50_processed.csv",
}
for name, path in FILES.items():
    print(f"{name:16} {'FOUND' if path.exists() else 'not found'} — {path}")

In [ ]:
def read_csv_if_exists(path):
    if not path.exists():
        return None
    try:
        return pd.read_csv(path)
    except Exception as exc:
        print(f"Could not read {path}: {exc}")
        return None

dfs = {name: read_csv_if_exists(path) for name, path in FILES.items()}
for name, df in dfs.items():
    if df is not None:
        print(f"{name:16}: {df.shape[0]:,} rows × {df.shape[1]:,} columns")

### Reading this section

A missing optional output is not necessarily a pipeline failure: output names can change between project versions, or a particular stage may not have been run. Confirm the filenames in the repository before interpreting an absent table as missing data.

The notebook uses actual column names from the loaded files wherever possible. The later cells attempt to identify common date, target, sentiment, and article-count fields, but they also print the detected fields so you can verify them.

## 4. Inspect schemas before analysis

In [ ]:
for name, df in dfs.items():
    if df is not None:
        print("\\n" + "="*90)
        print(name.upper(), "—", df.shape)
        display(df.head(5))
        print(df.dtypes.to_string())

In [ ]:
def find_col(df, candidates):
    if df is None:
        return None
    lower = {str(c).lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower:
            return lower[candidate.lower()]
    # fallback: substring match
    for c in df.columns:
        cl = str(c).lower()
        if any(token in cl for token in candidates):
            return c
    return None

model = dfs.get("model")
alignment = dfs.get("alignment")
daily_news = dfs.get("daily_news")
scored_news = dfs.get("scored_news")
daily_sentiment = dfs.get("daily_sentiment")
prices = dfs.get("prices")

DATE_CANDIDATES = ["date", "session_date", "trading_date", "timestamp", "datetime"]
date_cols = {name: find_col(df, DATE_CANDIDATES) for name, df in dfs.items() if df is not None}
print("Detected date columns:", date_cols)

TARGET_CANDIDATES = ["target_return_pct", "next_return_pct", "target_return", "future_return", "target"]
target_col = find_col(model, TARGET_CANDIDATES)
COUNT_CANDIDATES = ["article_count", "news_count", "n_articles", "headline_count", "article_count_total"]
count_col = find_col(daily_news, COUNT_CANDIDATES) or find_col(model, COUNT_CANDIDATES)
SENTIMENT_CANDIDATES = ["sentiment_mean", "mean_sentiment", "sentiment_score", "finbert_score", "sentiment"]
sentiment_col = find_col(daily_sentiment, SENTIMENT_CANDIDATES) or find_col(model, SENTIMENT_CANDIDATES)
print("Detected target:", target_col)
print("Detected article count:", count_col)
print("Detected sentiment:", sentiment_col)

## 5. Parse dates and establish the observation unit

A dataset can be row-level (one row per article) or session-level (one row per trading day). These are not interchangeable.

- In an **article-level** table, repeated dates are expected.
- In a **daily/session-level** table, repeated dates may indicate duplicate aggregation or multiple instruments.
- In a **model dataset**, the intended unit is generally one forecastable NIFTY session.

We parse likely date columns without silently dropping rows. Invalid dates are counted explicitly.

In [ ]:
for name, df in dfs.items():
    if df is None:
        continue
    dc = date_cols.get(name)
    if dc:
        df[dc] = pd.to_datetime(df[dc], errors="coerce", utc=False)
        invalid = df[dc].isna().sum()
        print(f"{name:16}: invalid/unparsed {dc} = {invalid:,} / {len(df):,}")

In [ ]:
def date_summary(df, date_col, label):
    if df is None or date_col is None:
        print(label, ": unavailable")
        return
    s = df[date_col].dropna()
    print({
        "table": label,
        "rows": len(df),
        "valid_dates": len(s),
        "unique_dates": s.nunique(),
        "first_date": s.min(),
        "last_date": s.max(),
        "duplicate_date_rows": int(s.duplicated().sum())
    })

for name, df in dfs.items():
    if df is not None:
        date_summary(df, date_cols.get(name), name)

### Interpretation checklist

- Is the time span long enough to include different market regimes?
- Does the dataset have one row per trading session or one row per calendar day?
- Are weekends and exchange holidays absent from the market table?
- Are missing dates genuine non-trading days, collection gaps, or a failed merge?
- Does the model dataset include the latest date as a feature row without a known next-session target? That can be appropriate for forecasting, but it must be excluded from historical scoring.

## 6. Data integrity: duplicates, missingness, and constant columns

In [ ]:
def quality_report(df, name):
    if df is None:
        return
    print(f"\\n{name}: {df.shape}")
    print("Exact duplicate rows:", int(df.duplicated().sum()))
    miss = df.isna().sum().sort_values(ascending=False)
    print("Columns with missing values:")
    display(miss[miss > 0].to_frame("missing_rows").assign(
        missing_pct=lambda x: (100*x["missing_rows"]/len(df)).round(2)
    ))
    nunique = df.nunique(dropna=False)
    print("Constant / single-value columns:", nunique[nunique <= 1].index.tolist())

for name, df in dfs.items():
    quality_report(df, name)

### Why missingness matters

Missing values can be structurally meaningful. For example, a no-news day may legitimately have zero article count, while a missing article count may mean the collection process failed. Do not replace all missing values with zero without distinguishing these cases.

A constant feature may be harmless metadata, but it contributes no information to a statistical model. A column that is constant because of an incorrect merge or parsing bug is a data-quality warning.

## 7. Market data: price and return structure

In [ ]:
market_df = prices if prices is not None else model
market_date = date_cols.get("prices") if prices is not None else date_cols.get("model")

if market_df is not None:
    print("Candidate market numeric columns:")
    display(market_df.select_dtypes(include=np.number).describe().T)

In [ ]:
def numeric_candidates(df, tokens):
    if df is None:
        return []
    return [c for c in df.columns if any(t in str(c).lower() for t in tokens)]

market_numeric = numeric_candidates(market_df, ["close", "return", "volatility", "volume", "open", "high", "low"])
print("Market-related columns:", market_numeric)

if market_df is not None and market_date:
    tmp = market_df.sort_values(market_date).copy()
    close_col = find_col(tmp, ["close", "nifty_close", "close_price"])
    ret_col = find_col(tmp, ["return_1d", "daily_return", "return_pct", "market_return"])
    if close_col:
        plt.figure(figsize=(14,4))
        plt.plot(tmp[market_date], tmp[close_col])
        plt.title("NIFTY close over time")
        plt.xlabel("Date"); plt.ylabel("Close"); plt.grid(alpha=.25); plt.show()
    if ret_col:
        plt.figure(figsize=(14,4))
        plt.plot(tmp[market_date], tmp[ret_col], linewidth=.8)
        plt.axhline(0, color="black", linewidth=.8)
        plt.title(f"Daily return series: {ret_col}")
        plt.xlabel("Date"); plt.ylabel("Return"); plt.grid(alpha=.25); plt.show()

### What to look for

1. **Trend and regime changes:** long-run price levels can rise while short-horizon returns remain centered near zero.
2. **Volatility clustering:** large moves often occur near other large moves. A rolling volatility feature may capture this.
3. **Outliers:** large returns may be real events, not errors. Verify against source data before removing them.
4. **Corporate/index adjustments:** confirm whether the input series is an index level, adjusted close, or another price field. The meaning of returns depends on that choice.

## 8. Return distribution and tail behavior

In [ ]:
if market_df is not None:
    ret_col = find_col(market_df, ["return_1d", "daily_return", "return_pct", "market_return"])
    if ret_col:
        r = pd.to_numeric(market_df[ret_col], errors="coerce").dropna()
        display(r.describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).to_frame("daily_return"))
        fig, axes = plt.subplots(1,2,figsize=(14,4))
        axes[0].hist(r, bins=60)
        axes[0].set_title("Daily return histogram")
        axes[0].set_xlabel(ret_col)
        axes[1].boxplot(r, vert=False)
        axes[1].set_title("Daily return boxplot")
        plt.show()

Financial returns are commonly heavy-tailed relative to a normal distribution. A few extreme observations can dominate mean-based summaries and model loss. Report median and quantiles alongside mean and standard deviation. Do not infer a normal distribution merely because a histogram looks roughly bell-shaped.

## 9. News collection: volume, source composition, and date coverage

In [ ]:
news_table = daily_news if daily_news is not None else alignment
news_date = date_cols.get("daily_news") if daily_news is not None else date_cols.get("alignment")
if news_table is not None:
    display(news_table.head())
    print("Columns:", news_table.columns.tolist())

In [ ]:
if news_table is not None:
    ccol = find_col(news_table, COUNT_CANDIDATES)
    if ccol:
        counts = pd.to_numeric(news_table[ccol], errors="coerce")
        display(counts.describe(percentiles=[.5,.75,.9,.95,.99]).to_frame("article_count"))
        plt.figure(figsize=(14,4))
        plt.hist(counts.dropna(), bins=50)
        plt.title("Distribution of daily article counts")
        plt.xlabel(ccol); plt.ylabel("Number of observations"); plt.show()
        if news_date:
            t = news_table[[news_date, ccol]].dropna().sort_values(news_date)
            plt.figure(figsize=(14,4))
            plt.plot(t[news_date], t[ccol], linewidth=.8)
            plt.title("Daily news volume over time")
            plt.xlabel("Date"); plt.ylabel("Articles"); plt.grid(alpha=.25); plt.show()

### News volume interpretation

Article counts can proxy for attention or event intensity, but collection volume is also affected by:
- query terms and RSS provider behavior;
- duplicate/syndicated headlines;
- source availability and rate limits;
- changes in collection code;
- sampling caps or deduplication rules.

A spike in the dataset is not automatically a spike in all financial news. Treat volume as **observed collection volume**, not a complete census of market information.

## 10. Article-level inspection and duplication

In [ ]:
if scored_news is not None:
    print("Scored headline columns:", scored_news.columns.tolist())
    text_col = find_col(scored_news, ["title", "headline", "article_title"])
    source_col = find_col(scored_news, ["source", "publisher", "domain"])
    if text_col:
        print("Rows:", len(scored_news))
        print("Unique titles:", scored_news[text_col].nunique(dropna=True))
        print("Exact duplicate titles:", int(scored_news[text_col].duplicated().sum()))
        display(scored_news[[c for c in [date_cols.get("scored_news"), source_col, text_col] if c]].head(15))
    if source_col:
        display(scored_news[source_col].value_counts(dropna=False).head(20).to_frame("articles"))

Headline duplication deserves special attention. Multiple outlets can publish near-identical versions of the same wire story. If all are counted independently, a single event may receive disproportionate weight. Deduplication should be documented and should not accidentally remove genuinely distinct articles.

## 11. Sentiment scores: distribution and calibration cautions

In [ ]:
sent_table = daily_sentiment if daily_sentiment is not None else scored_news
if sent_table is not None:
    print("Sentiment-related columns:", numeric_candidates(sent_table, ["sentiment", "positive", "negative", "neutral", "score"]))
    display(sent_table.select_dtypes(include=np.number).describe().T)

In [ ]:
if sent_table is not None:
    sent_col = find_col(sent_table, SENTIMENT_CANDIDATES)
    if sent_col:
        vals = pd.to_numeric(sent_table[sent_col], errors="coerce").dropna()
        plt.figure(figsize=(10,4))
        plt.hist(vals, bins=50)
        plt.title(f"Distribution of {sent_col}")
        plt.xlabel(sent_col); plt.ylabel("Count"); plt.show()
        print("n =", len(vals), "mean =", vals.mean(), "median =", vals.median(), "std =", vals.std())

### Important sentiment distinction

FinBERT produces model-based sentiment estimates from text. These are not direct measurements of investor belief, market impact, or the magnitude of an expected return. A positive headline may be already priced in, irrelevant to NIFTY, or offset by other news. Sentiment aggregation can also hide disagreement: a daily average near zero may represent uniformly neutral news or a mixture of very positive and very negative headlines.

If the pipeline stores positive/negative/neutral probabilities, inspect all three and verify that the aggregate feature definition is documented. Avoid interpreting a raw model score as a calibrated probability of market movement.

## 12. Sampling audit and coverage changes

In [ ]:
audit = dfs.get("sampling_audit")
if audit is not None:
    display(audit.head(20))
    print(audit.shape, audit.columns.tolist())
else:
    print("No sampling audit file found.")

Sampling caps can create nonuniform observation density. For example, if high-volume months are capped, the sampled article set may underrepresent the most news-intensive periods. Compare raw and retained counts if both are available. Sampling must be applied consistently and should not use future target values.

## 13. Join/alignment audit: dates, row counts, and unmatched records

In [ ]:
if alignment is not None:
    print("Alignment table:", alignment.shape)
    print("Potential date/key columns:", [c for c in alignment.columns if any(t in str(c).lower() for t in ["date","time","session","ticker","symbol"])])
    display(alignment.head(10))

In [ ]:
# If article-level alignment has a date and a market/session date, compare them.
if alignment is not None:
    possible_dates = [c for c in alignment.columns if "date" in str(c).lower() or "time" in str(c).lower()]
    print("Possible temporal columns:", possible_dates)
    for c in possible_dates:
        parsed = pd.to_datetime(alignment[c], errors="coerce")
        print(c, "valid:", parsed.notna().sum(), "unique:", parsed.nunique(), "range:", parsed.min(), "to", parsed.max())

### Alignment questions to answer before modeling

- Is each article assigned to the session whose forecast cutoff it precedes?
- What happens to after-close headlines?
- How are weekends and exchange holidays handled?
- Are article timestamps stored with a timezone, and is conversion to India Standard Time explicit?
- Is the target next-session return joined by the correct trading-session sequence rather than calendar-day arithmetic?
- Are external-market features lagged when their close occurs after the NIFTY cutoff?

A date-only merge cannot answer these questions. The intended information cutoff must be explicit in the pipeline.

## 14. Feature availability and look-ahead-bias audit

In [ ]:
# This is a manual-audit scaffold. Populate these values from source documentation
# and actual timestamps. Do not assume a same-date value was available at forecast time.
availability = pd.DataFrame([
    {"feature_group":"NIFTY same-session close/return", "availability":"At/after NIFTY close", "safe_for_same-session forecast?":"Depends on forecast cutoff"},
    {"feature_group":"USD/INR daily close", "availability":"Verify exact source timestamp; may be after NIFTY close", "safe_for_same-session forecast?":"Potentially unsafe; lag if later"},
    {"feature_group":"Gold / Brent daily return", "availability":"Verify source market/session and timestamp", "safe_for_same-session forecast?":"Unknown until verified"},
    {"feature_group":"News headline", "availability":"Use publication timestamp", "safe_for_same-session forecast?":"Only if published before cutoff"},
    {"feature_group":"Next-session return target", "availability":"After next session closes", "safe_for_same-session forecast?":"Target only; never a feature"},
])
display(availability)

**Leakage rule:** At prediction time, every feature must be computable from information genuinely available by that time. If a feature is not available by the cutoff, lag it or remove it. A model can achieve apparently strong historical results by using information that would not have existed when the forecast was made.

## 15. Model dataset: one row per forecast origin

In [ ]:
if model is not None:
    print("Model dataset shape:", model.shape)
    display(model.head())
    if target_col:
        target = pd.to_numeric(model[target_col], errors="coerce")
        print("Target non-null:", target.notna().sum(), "/", len(target))
        display(target.describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).to_frame(target_col))

In [ ]:
if model is not None and target_col:
    y = pd.to_numeric(model[target_col], errors="coerce")
    fig, axes = plt.subplots(1,2,figsize=(14,4))
    axes[0].hist(y.dropna(), bins=60)
    axes[0].set_title("Forecast target distribution")
    axes[0].set_xlabel(target_col)
    axes[1].boxplot(y.dropna(), vert=False)
    axes[1].set_title("Target outliers")
    plt.show()

### Target definition must be unambiguous

Document whether the target is:
- next-session close-to-close return;
- next-session open-to-close return;
- next-session direction;
- or a thresholded large-move label.

The denominator, units (fraction vs percent), session convention, and handling of the final row must be explicit. A column named `target` is not enough to establish its meaning.

## 16. Missingness over time

In [ ]:
if model is not None and date_cols.get("model"):
    dcol = date_cols["model"]
    miss = model.isna().groupby(model[dcol].dt.to_period("M")).mean()
    if not miss.empty:
        ax = miss.plot(figsize=(15,5), legend=False)
        ax.set_title("Monthly missingness rate by feature")
        ax.set_xlabel("Month"); ax.set_ylabel("Missing fraction")
        plt.show()

A feature can have low overall missingness but concentrated gaps during particular market periods. That matters if missingness correlates with volatility or news intensity. Missing-data handling should be learned on training data only, especially in a walk-forward evaluation.

## 17. Correlation analysis (descriptive, not causal)

In [ ]:
if model is not None:
    num = model.select_dtypes(include=np.number)
    # Remove nearly constant columns for readability
    num = num.loc[:, num.nunique(dropna=True) > 1]
    if num.shape[1] >= 2:
        corr = num.corr()
        if HAS_SEABORN:
            plt.figure(figsize=(min(18, 0.7*len(corr.columns)+5), min(14, 0.7*len(corr.columns)+4)))
            sns.heatmap(corr, cmap="vlag", center=0, vmin=-1, vmax=1)
            plt.title("Numeric feature correlation matrix")
            plt.show()
        else:
            display(corr.round(2))

Correlation is useful for spotting redundant features, unexpected signs, and possible leakage. It does **not** establish causation or out-of-sample predictive value. Correlations can also be unstable across regimes and can be distorted by trends, common time effects, or data leakage.

## 18. News/sentiment versus market outcomes

In [ ]:
# Merge only if both tables have a usable date column. This is exploratory and
# should not replace the pipeline's forecast-cutoff-aware alignment.
if model is not None and daily_sentiment is not None:
    mdcol = date_cols.get("model")
    sdcol = date_cols.get("daily_sentiment")
    if mdcol and sdcol:
        left = model.copy()
        right = daily_sentiment.copy()
        left["_date_key"] = pd.to_datetime(left[mdcol], errors="coerce").dt.date
        right["_date_key"] = pd.to_datetime(right[sdcol], errors="coerce").dt.date
        merged = left.merge(right, on="_date_key", how="left", suffixes=("_model","_sent"))
        print("Model rows:", len(left), "merged rows:", len(merged))
        print("Rows with no sentiment match:", merged.filter(regex="_sent$").isna().all(axis=1).sum() if len(merged.columns) else "inspect")
        display(merged.head())
    else:
        print("Date columns not detected; inspect schemas and set the keys manually.")
else:
    print("Need both model dataset and daily sentiment table for this exploratory merge.")

If a relationship appears, test it with a properly specified chronological model. Do not choose a feature because it has the strongest full-sample correlation and then evaluate it on that same sample. That is a form of selection bias.

## 19. Compare feature groups without leaking future information

In [ ]:
if model is not None:
    print("Numeric columns:")
    print(model.select_dtypes(include=np.number).columns.tolist())
    print("\\nPotential target/label columns:")
    print([c for c in model.columns if any(t in str(c).lower() for t in ["target","label","future","next","direction"])])

A useful experimental design compares at least:

1. **Naive baseline:** predict zero return, or the expanding historical mean.
2. **Market-only model:** lagged returns, volatility, and other cutoff-safe market features.
3. **Market + news model:** same market features plus news volume and sentiment.

The comparison must use the same forecast dates, target, training window, and evaluation metrics. The news model should not get a more favorable split or more tuning opportunities than the baseline.

## 20. Chronological validation and metric interpretation

For time series, random shuffling mixes past and future. Use expanding-window or rolling-window evaluation:

- Train only on observations before the forecast date.
- Apply all fitted transformations using training data only.
- Predict the next observation (or next block).
- Advance the cutoff and repeat.

Recommended metrics:
- **MAE / RMSE:** return error, in the same units as the target.
- **R²:** comparison to the test-set mean; negative values indicate worse squared error than that reference.
- **Direction accuracy:** proportion of signs predicted correctly; weakly informative when classes are imbalanced.
- **Brier score:** probability quality for binary direction; lower is better, with calibration and class prevalence considered.
- **Trading metrics:** require explicit position sizing, transaction costs, slippage, risk controls, and a realistic execution assumption. Do not infer profitability from accuracy alone.

## 21. Audit the latest forecast and forecast history

In [ ]:
forecast_path = DATA_DIR / "processed" / "latest_forecast.csv"
outcomes_path = DATA_DIR / "processed" / "forecast_outcome_history.csv"
latest_forecast = read_csv_if_exists(forecast_path)
outcomes = read_csv_if_exists(outcomes_path)
if latest_forecast is not None:
    print("Latest forecast")
    display(latest_forecast)
if outcomes is not None:
    print("Recorded outcomes:", outcomes.shape)
    display(outcomes.tail(10))
else:
    print("No forecast outcome history found.")

A small number of recorded forecasts cannot establish model skill. For example, 4 correct directions out of 4 is still a very uncertain estimate. Report the denominator, dates, and baseline comparison every time. Prospective outcomes should be recorded automatically and evaluated without changing the model after seeing each result.

## 22. Limitations and threats to validity

### Data coverage
RSS/search-derived headlines are incomplete and query-dependent. Coverage may vary over time and across topics. Duplicate stories and missing publication timestamps can distort aggregates.

### Measurement
FinBERT sentiment is a text-model output, not a direct measure of market expectations. Headlines omit article context and may be ambiguous, sarcastic, or unrelated to the NIFTY constituents.

### Temporal alignment
Calendar-date joins can leak information. Forecast cutoff, timezone, publication time, exchange holidays, and external market close times must be audited explicitly.

### Statistical uncertainty
Financial returns are noisy, nonstationary, and heavy-tailed. A small test period or a few successful forecasts can produce misleading results.

### Model selection
Repeatedly inspecting the test period and changing features creates implicit overfitting. Keep a final untouched period or use nested chronological validation.

### Economic interpretation
A statistically lower error does not automatically imply a profitable strategy. Trading costs, turnover, liquidity, and risk must be modeled separately.

## 23. EDA findings template

Complete this section after running the notebook. Do not fill it with assumptions.

- **Coverage:** [start/end date, number of sessions, number of headlines]
- **Data quality:** [duplicate rows, missingness, invalid dates]
- **Market behavior:** [return distribution, volatility clustering, outliers]
- **News behavior:** [volume variation, source concentration, sampling effects]
- **Sentiment behavior:** [score distribution, missingness, aggregation definition]
- **Alignment:** [cutoff, timezone, external-market lag decisions]
- **Potential concerns:** [leakage, selection bias, coverage changes]
- **Next experiment:** [one hypothesis, one comparison, one chronological protocol]

## 24. Final interpretation guardrails

Before presenting conclusions, check:

- Did the analysis use the exact target definition and units?
- Were all features available at the forecast cutoff?
- Were duplicate headlines and sampling rules understood?
- Were exploratory correlations kept separate from predictive claims?
- Was model evaluation chronological and leakage-safe?
- Were all baselines evaluated on identical dates?
- Are uncertainty and small sample sizes reported?
- Does the conclusion say what the evidence supports—and no more?

**Bottom line:** this notebook is a diagnostic instrument. It can expose data problems, characterize the dataset, and generate hypotheses. It cannot by itself prove that news sentiment predicts NIFTY returns or that the system is profitable.